# Étape 4 — Variations stylistiques et signal de genre

**Objectif :** Examiner l'existence d'un signal stylistique associé au sexe des candidats,  
indépendamment de l'appartenance partisane.

**Pipeline :**
1. Extraction de features stylistiques interprétables par document
2. Analyse descriptive et tests statistiques (contrôle du parti)
3. Régressions logistiques pour isoler l'effet du sexe
4. Classification supervisée (évaluation de la prédictibilité du genre)

> ⚠️ Les résultats sont interprétés avec prudence compte tenu  
> du déséquilibre important homme/femme dans le corpus (≈ 88 % / 12 %).

## 0. Imports

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import spacy
from scipy import stats
from collections import Counter

# Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils import resample

import statsmodels.formula.api as smf
import statsmodels.api as sm

warnings.filterwarnings("ignore")

DATA_DIR    = Path("../data")
PROC_DIR    = DATA_DIR / "processed"
FIGURES_DIR = Path("../figures")

print("Librairies chargées ✓")

## 1. Chargement des données

In [ ]:
df = pd.read_csv(PROC_DIR / "corpus_1993.csv")

# Garder uniquement homme / femme (exclure non déterminé)
df_gendered = df[df["titulaire-sexe"].isin(["homme", "femme"])].copy().reset_index(drop=True)

print(f"Corpus total       : {len(df):,}")
print(f"Corpus avec genre  : {len(df_gendered):,}")
print(f"  Hommes           : {(df_gendered['titulaire-sexe']=='homme').sum():,}")
print(f"  Femmes           : {(df_gendered['titulaire-sexe']=='femme').sum():,}")
print(f"  Ratio F/(H+F)    : {(df_gendered['titulaire-sexe']=='femme').mean()*100:.1f}%")

## 2. Extraction des features stylistiques

On calcule pour chaque document les indicateurs suivants :

| Feature | Description |
|---------|-------------|
| `n_sentences` | Nombre de phrases |
| `mean_sent_len` | Longueur moyenne des phrases (mots) |
| `std_sent_len` | Écart-type de la longueur des phrases |
| `mean_word_len` | Longueur moyenne des mots (caractères) |
| `msttr` | Richesse lexicale (MSTTR fenêtre 100) |
| `pron_1sg_rate` | Fréquence des pronoms 1ère pers. sing. (je, me, moi, mon, ma) |
| `pron_1pl_rate` | Fréquence des pronoms 1ère pers. plur. (nous, notre, nos) |
| `modal_rate` | Fréquence des verbes modaux (devoir, pouvoir, falloir, vouloir) |
| `adj_rate` | Fréquence relative des adjectifs |
| `flesh_fr` | Score de lisibilité Flesch adapté au français |

> ⏱ L'extraction spaCy prend ~3 min sur le corpus complet.

In [ ]:
try:
    nlp = spacy.load("fr_core_news_md")
except OSError:
    nlp = spacy.load("fr_core_news_sm")

PRON_1SG = {"je", "j", "me", "moi", "mon", "ma", "mes"}
PRON_1PL = {"nous", "notre", "nos", "on"}
MODAUX   = {"devoir", "pouvoir", "falloir", "vouloir", "savoir", "oser", "osons"}

def extract_style_features(text: str) -> dict:
    doc = nlp(text)
    tokens_alpha = [t for t in doc if t.is_alpha]
    n_tokens = len(tokens_alpha) or 1

    # Découpage en phrases
    sentences  = list(doc.sents)
    sent_lens  = [len([t for t in s if t.is_alpha]) for s in sentences]
    sent_lens  = [l for l in sent_lens if l > 0]

    # Pronoms
    lemmas_lower = [t.lemma_.lower() for t in tokens_alpha]
    p1sg  = sum(1 for l in lemmas_lower if l in PRON_1SG)
    p1pl  = sum(1 for l in lemmas_lower if l in PRON_1PL)
    modal = sum(1 for l in lemmas_lower if l in MODAUX)
    n_adj = sum(1 for t in tokens_alpha if t.pos_ == "ADJ")

    # Longueur des mots
    word_lens = [len(t.text) for t in tokens_alpha]

    # Flesch adapté français (approximation) :
    # 207 - 1.015 × (mots/phrases) - 73.6 × (syllabes/mots)
    avg_sl  = np.mean(sent_lens) if sent_lens else 1
    # Estimation syllabes : voyelles consécutives
    def count_syllables(word):
        return max(1, len([c for c in word.lower() if c in "aeéèêëiîïoôuùûy"]))
    avg_syl = np.mean([count_syllables(t.text) for t in tokens_alpha]) if tokens_alpha else 1
    flesch  = 207 - 1.015 * avg_sl - 73.6 * avg_syl

    return {
        "n_sentences"   : len(sent_lens),
        "mean_sent_len" : np.mean(sent_lens) if sent_lens else 0,
        "std_sent_len"  : np.std(sent_lens)  if len(sent_lens) > 1 else 0,
        "mean_word_len" : np.mean(word_lens) if word_lens else 0,
        "pron_1sg_rate" : p1sg  / n_tokens * 100,
        "pron_1pl_rate" : p1pl  / n_tokens * 100,
        "modal_rate"    : modal / n_tokens * 100,
        "adj_rate"      : n_adj / n_tokens * 100,
        "flesch_fr"     : flesch,
    }

print("Extraction des features stylistiques en cours...")
style_rows = [extract_style_features(text) for text in df_gendered["clean_text_lda"]]
df_style   = pd.DataFrame(style_rows)

# Ajouter le MSTTR depuis le CSV (déjà calculé à l'étape 1)
df_style["msttr"] = df_gendered["msttr"].values

# Concaténer
STYLE_FEATURES = list(df_style.columns)
for col in STYLE_FEATURES:
    df_gendered[col] = df_style[col].values

print(f"Features extraites ({len(STYLE_FEATURES)}) : {STYLE_FEATURES}")

## 3. Analyse descriptive — comparaison homme / femme

In [ ]:
desc = (
    df_gendered.groupby("titulaire-sexe")[STYLE_FEATURES]
    .agg(["mean", "std", "median"])
    .round(3)
)
print("Statistiques descriptives par sexe :")
display(desc)

### 3.1 Visualisation des distributions

Violin plots pour chaque feature, afin de comparer les distributions  
homme / femme au-delà des simples moyennes.

In [ ]:
n_feat = len(STYLE_FEATURES)
ncols  = 3
nrows  = (n_feat + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
axes = axes.flatten()

palette = {"homme": "#4C8BE2", "femme": "#E24C8B"}

for ax, feat in zip(axes, STYLE_FEATURES):
    sns.violinplot(
        data=df_gendered, x="titulaire-sexe", y=feat,
        palette=palette, ax=ax, inner="quartile", cut=0
    )
    ax.set_title(feat, fontsize=11)
    ax.set_xlabel("")

for ax in axes[n_feat:]:
    ax.set_visible(False)

plt.suptitle("Distributions stylistiques par sexe", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_style_violins.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.2 Tests de Mann-Whitney U

Test non paramétrique (robuste aux distributions non normales et aux outliers)  
comparant chaque feature entre hommes et femmes.  
Correction de Bonferroni appliquée pour les tests multiples (α ajusté = 0.05 / n_features).

In [ ]:
alpha_bonf = 0.05 / len(STYLE_FEATURES)
results = []

for feat in STYLE_FEATURES:
    h = df_gendered[df_gendered["titulaire-sexe"] == "homme"][feat].dropna()
    f = df_gendered[df_gendered["titulaire-sexe"] == "femme"][feat].dropna()
    stat, p = stats.mannwhitneyu(h, f, alternative="two-sided")

    # Effect size r = Z / sqrt(N)
    n = len(h) + len(f)
    z = (stat - len(h)*len(f)/2) / np.sqrt(len(h)*len(f)*(n+1)/12)
    r = abs(z) / np.sqrt(n)

    results.append({
        "feature"   : feat,
        "mean_H"    : h.mean(), "mean_F"  : f.mean(),
        "diff_F_H"  : f.mean() - h.mean(),
        "U_stat"    : stat,  "p_value" : p,
        "effect_r"  : r,
        "significant": p < alpha_bonf
    })

df_tests = pd.DataFrame(results).sort_values("p_value")
print(f"Seuil Bonferroni : α = {alpha_bonf:.4f}\n")
display(df_tests.round(5))

### 3.3 Contrôle de l'appartenance partisane — Régressions OLS

Pour éviter les effets de confusion (certains partis comptant plus de femmes  
et ayant peut-être un style propre), on modélise chaque feature par :

```
feature ~ titulaire_sexe + C(titulaire_soutien_simplifie)
```

On regarde le coefficient de `titulaire_sexe` : est-il significatif  
une fois les différences inter-partis contrôlées ?

In [ ]:
df_reg = df_gendered.copy()
df_reg["sexe_bin"] = (df_reg["titulaire-sexe"] == "femme").astype(int)
df_reg.columns = df_reg.columns.str.replace(r"[^a-zA-Z0-9_]", "_", regex=True)

reg_results = []
for feat in STYLE_FEATURES:
    feat_safe = feat.replace("-", "_")
    formula = f"{feat_safe} ~ sexe_bin + C(titulaire_soutien_simplifie)"
    try:
        model  = smf.ols(formula=formula, data=df_reg).fit()
        coef   = model.params["sexe_bin"]
        pval   = model.pvalues["sexe_bin"]
        ci_low, ci_hi = model.conf_int().loc["sexe_bin"]
        reg_results.append({
            "feature": feat, "coef_sexe": coef,
            "ci_low": ci_low, "ci_high": ci_hi,
            "p_value": pval, "r2": model.rsquared,
            "significant": pval < 0.05
        })
    except Exception as e:
        print(f"  [skip] {feat} : {e}")

df_reg_res = pd.DataFrame(reg_results).sort_values("p_value")
print("Effet du sexe après contrôle du parti (OLS) :\n")
display(df_reg_res.round(4))

In [ ]:
# Forest plot des coefficients
fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#E24C8B" if s else "#888888" for s in df_reg_res["significant"]]

ax.barh(df_reg_res["feature"], df_reg_res["coef_sexe"], color=colors, alpha=0.8)
ax.errorbar(
    df_reg_res["coef_sexe"], df_reg_res["feature"],
    xerr=[df_reg_res["coef_sexe"] - df_reg_res["ci_low"],
          df_reg_res["ci_high"] - df_reg_res["coef_sexe"]],
    fmt="none", color="black", capsize=4, lw=1.5
)
ax.axvline(0, color="black", lw=0.8, linestyle="--")
ax.set_xlabel("Coefficient (femme vs homme, contrôle parti)")
ax.set_title("Effect du sexe sur les features stylistiques\n(rose = p < 0.05)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_style_regression_coefs.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Classification supervisée — Prédire le genre

On entraîne une régression logistique pour prédire le genre du candidat  
à partir des features stylistiques, en validation croisée stratifiée (5 folds).

> ⚠️ Le déséquilibre homme/femme (≈ 88 % / 12 %) est géré par  
> `class_weight='balanced'` et évalué sur le F1-score macro.

In [ ]:
X = df_gendered[STYLE_FEATURES].values
y = (df_gendered["titulaire-sexe"] == "femme").astype(int).values

# Pipeline : normalisation + régression logistique
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for metric in ("accuracy", "f1_macro", "roc_auc"):
    scores = cross_val_score(clf, X, y, cv=cv, scoring=metric)
    print(f"{metric:15s} : {scores.mean():.4f}  ± {scores.std():.4f}")

In [ ]:
# Baseline : toujours prédire la classe majoritaire
baseline_acc = (y == 0).mean()
print(f"\nBaseline (majorité) : {baseline_acc:.4f}")

# Entraînement final + matrice de confusion
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print("\nRapport de classification (test 20%) :")
print(classification_report(y_te, y_pred, target_names=["homme", "femme"]))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_te, y_pred, display_labels=["homme", "femme"],
    colorbar=False, ax=ax
)
ax.set_title("Matrice de confusion (test 20%)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Importance des features (coefficients)
coefs = clf.named_steps["lr"].coef_[0]
feat_imp = pd.Series(coefs, index=STYLE_FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.plot.barh(ax=ax, color=["#E24C8B" if c > 0 else "#4C8BE2" for c in feat_imp])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Coefficient (positif → prédit femme)")
ax.set_title("Importance des features stylistiques pour la prédiction du genre")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Synthèse des résultats

In [ ]:
print("=" * 60)
print("SYNTHÈSE — Analyse stylistique par genre")
print("=" * 60)

sig_mw   = df_tests[df_tests["significant"]]["feature"].tolist()
sig_ols  = df_reg_res[df_reg_res["significant"]]["feature"].tolist()
sig_both = set(sig_mw) & set(sig_ols)

print(f"\nFeatures différentes H/F (Mann-Whitney, Bonferroni) : {sig_mw}")
print(f"Features sig. après contrôle parti (OLS)            : {sig_ols}")
print(f"Features robustes (sig. dans les deux tests)         : {sig_both}")

print(f"\nClassification (CV 5-fold) :")
print(f"  AUC moyen : voir cellule précédente")
print(f"  Baseline  : {baseline_acc:.3f}")

## 6. Sauvegarde

In [ ]:
STYLE_COLS = ["id", "titulaire-sexe", "titulaire-soutien-simplifie"] + STYLE_FEATURES
df_gendered[STYLE_COLS].to_csv(PROC_DIR / "style_features_1993.csv", index=False)
print("✓ Sauvegardé :", PROC_DIR / "style_features_1993.csv")